# Introduction

Previous experiments on the Maji Ndogo agricultural dataset showed that nonlinear and ensemble models substantially outperformed the initial linear models. In particular, Decision Tree, Random Forest, and Gradient Boosting captured complex relationships between environmental, soil, weather, and agricultural variables and crop yield. This experiment investigates whether combining these strong models through stacking regression can further improve predictive performance.

# Objective

To evaluate whether a StackingRegressor combining the tuned Decision Tree, Random Forest, and Gradient Boosting models can achieve better predictive performance than the individual models.

# Hypothesis

### Null Hypothesis (H₀):
Combining the Decision Tree, Random Forest, and Gradient Boosting models through stacking will not significantly improve predictive performance compared with the best individual model.

### Alternative Hypothesis (H₁):
Combining the models through stacking will improve predictive performance by leveraging the complementary strengths of the individual models.

In [ ]:
cd ..

In [ ]:
import numpy as np 
import pandas as pd

from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
from model_pipelines.ensemble_models import EnsembleStacking, TreePipeline
from src.preprocessing import ModelPreprocessors


In [ ]:
mp = ModelPreprocessors()
mp.prepare_data()

In [ ]:
tree = TreePipeline()
tree.prepare_data()
tree.tree_feature_transformation()


In [ ]:
ensemble = EnsembleStacking()
ensemble.prepare_data()
ensemble.feature_transfromer()


In [ ]:
X_train, X_test, y_train, y_test = mp.X_y_train_test_split()

In [ ]:
# This parameters were tested and validated through random search cross valiadation. Refer to each algorithmn notebook

gradient_params = {
    "n_estimators": 661,
    "learning_rate": 0.168307,
    "max_depth": 4,
    "min_samples_split": 18,
    "min_samples_leaf": 14,
    "subsample": 0.865009
}

tree_params = {
    "min_samples_split": 4,
    "max_depth": 36
}

forest_params = {
    "max_depth": 33,
    "min_samples_split": 1,
    "n_estimators": 444,
    "max_features": 1
}

In [ ]:
tree_pipeline = tree.decision_tree_pipeline(**tree_params)
forest_pipeline = ensemble.random_forest_pipeline(**forest_params)
gradient_pipeline = ensemble.gradient_boosting_pipeline(**gradient_params)

In [ ]:
def base_stacking_regressor(X_train, X_test, y_train):
    stacking_model = ensemble.stacking_regressor_pipeline()
    fitted_model = stacking_model.fit(X_train, y_train)
    y_stack_pred = fitted_model.predict(X_test)
    return fitted_model, y_stack_pred


fitted_model, y_stack_pred = base_stacking_regressor(X_train, X_test, y_train)
display(fitted_model)

In [12]:
def stacking_evaluation(y_test, y_stack_pred):
    rmse = root_mean_squared_error(y_test, y_stack_pred)
    r2 = r2_score(y_test, y_stack_pred)
    return rmse, r2

rmse, r2 = stacking_evaluation(y_test, y_stack_pred)
print(f"RMSE: {rmse:.4f}")
print(f"r2 score: {r2:.4f}")

RMSE: 0.0218
r2 score: 0.9643


## Model Performance Comparison

| Model | RMSE | R² |
|---|---:|---:|
| OLS / Linear Regression | 0.0678 | 0.6553 |
| RidgeCV | 0.0678 | 0.6552 |
| LassoCV | 0.0699 | 0.6333 |
| Polynomial Regression | 0.0589 | 0.7396 |
| Decision Tree | 0.0322 | 0.9223 |
| RandomForest | 0.0208 | 0.9674 |
| GradientBoostingRegressor | 0.0148 | 0.9836 |
| StackingRegressor | 0.0218 | 0.9643 |

# Conclusion

The stacking experiment combined the tuned Decision Tree, Random Forest, and Gradient Boosting models to determine whether their combined predictions could outperform the individual models. Although the stacking model achieved strong performance (**RMSE: 0.0218, R²: 0.9643**), it did not surpass the tuned Gradient Boosting model (**RMSE: 0.0148, R²: 0.9836**).

This result shows that stacking does not automatically improve predictive performance. In this case, the additional complexity of combining multiple models did not provide an advantage over the already highly effective Gradient Boosting model. Therefore, **Gradient Boosting remains the best-performing model for predicting Standard Yield in the Maji Ndogo dataset**.
